# 🔊 NEURO BAND — Voice Add-On
### AI Health Brief → Spoken Out Loud (Web Speech API)

---
**Add these 2 cells AFTER Cell 7 (AI Health) in your existing notebook.**

- Uses the **browser's built-in Web Speech API** — completely FREE, no API key needed
- Works in **Chrome** and **Edge** best (recommended)
- **Cell 7-A** — Voice Engine Setup (run once per session)
- **Cell 7-B** — Speak the Health Brief (run after Cell 7)

> 💡 Also works inside Cell 8's dashboard with a play button!

In [ ]:
# ============================================================
# CELL 7-A — VOICE ENGINE SETUP
# Uses Web Speech API (FREE, built into Chrome/Edge)
# Run once per session, before Cell 7-B
# ============================================================

from IPython.display import display, HTML, Javascript
import json

# ─── Voice settings — tweak these to your preference ────────
VOICE_SETTINGS = {
    'rate'  : 0.92,   # Speed: 0.5 (slow) → 1.0 (normal) → 1.5 (fast)
    'pitch' : 1.05,   # Pitch: 0.5 (deep) → 1.0 (normal) → 2.0 (high)
    'volume': 1.0,    # Volume: 0.0 (mute) → 1.0 (max)
    # Voice name — best free voices by browser:
    # Chrome on Windows : 'Google US English'  or 'Microsoft David'
    # Chrome on Mac     : 'Samantha'           or 'Karen'
    # Chrome on Linux   : 'Google UK English Female'
    # Leave as '' to use browser default
    'voice_name': 'Google US English'
}

# ─── Inject the voice engine into the Colab browser ─────────
display(HTML("""
<div id='nb-voice-status'
     style='font-family:sans-serif;padding:10px 16px;border-radius:8px;
            background:#1a1a2e;color:#a0a8ff;font-size:.85em;display:inline-block;'>
  🔊 Voice engine initialising…
</div>
<script>
// ── Store settings globally in browser window ────────────────
window.nbVoiceSettings = {
  rate      : 0.92,
  pitch     : 1.05,
  volume    : 1.0,
  voiceName : 'Google US English'
};

// ── Core speak function ──────────────────────────────────────
window.nbSpeak = function(text, onDone) {
  if (!window.speechSynthesis) {
    alert('Speech not supported. Please use Chrome or Edge.');
    return;
  }
  window.speechSynthesis.cancel();

  // Clean the text: remove HTML tags and special chars
  var clean = text
    .replace(/<[^>]*>/g, ' ')
    .replace(/[🧬📚✅🌤️💧🧠☀️💨🌡️⚡🧠🔴🟡🟢🔊]/gu, '')
    .replace(/[\u{1F300}-\u{1FFFF}]/gu, '')
    .replace(/&nbsp;/g, ' ')
    .replace(/\s+/g, ' ')
    .trim();

  var utter  = new SpeechSynthesisUtterance(clean);
  var cfg    = window.nbVoiceSettings;
  utter.rate   = cfg.rate;
  utter.pitch  = cfg.pitch;
  utter.volume = cfg.volume;

  // Try to find the requested voice
  var voices = window.speechSynthesis.getVoices();
  if (cfg.voiceName && voices.length > 0) {
    var match = voices.find(v => v.name.includes(cfg.voiceName));
    if (!match) match = voices.find(v => v.lang.startsWith('en'));
    if (match) utter.voice = match;
  }

  utter.onend   = function() { if (onDone) onDone(); };
  utter.onerror = function(e) { console.warn('TTS error:', e.error); };

  window.speechSynthesis.speak(utter);
};

// ── Pause / Resume / Stop ───────────────────────────────────
window.nbPause  = function() { window.speechSynthesis.pause();  };
window.nbResume = function() { window.speechSynthesis.resume(); };
window.nbStop   = function() { window.speechSynthesis.cancel(); };

// ── Wait for voices to load (Chrome loads them async) ───────
function initVoice() {
  var s = document.getElementById('nb-voice-status');
  if (window.speechSynthesis) {
    if (window.speechSynthesis.onvoiceschanged !== undefined) {
      window.speechSynthesis.onvoiceschanged = function() {
        var v = window.speechSynthesis.getVoices();
        s.innerHTML = '✅ Voice engine ready! Found ' + v.length + ' voice(s). Run Cell 7-B to speak.';
        s.style.color = '#2ed573';
      };
    }
    // Also trigger immediately in case voices are already loaded
    window.speechSynthesis.getVoices();
    s.innerHTML = '✅ Voice engine ready! Run Cell 7-B to speak the health brief.';
    s.style.color = '#2ed573';
  } else {
    s.innerHTML = '❌ Speech not supported. Please use Chrome or Edge browser.';
    s.style.color = '#ff4757';
  }
}
initVoice();
</script>
"""))

print('\n✅ Voice engine injected into browser!')
print('   Settings:', VOICE_SETTINGS)
print()
print('📌 TIPS:')
print('   • Use Chrome or Edge for best voice quality')
print('   • If you hear no sound, check your system volume')
print('   • Change VOICE_SETTINGS above and re-run to adjust')
print()
print('👉 Now run Cell 7-B to speak the AI health brief!')

In [ ]:
# ============================================================
# CELL 7-B — SPEAK THE AI HEALTH BRIEF
# Full interactive player with play/pause/stop/replay
# Run AFTER Cell 7 (AI Health Recommendations)
# ============================================================

import json

def speak_health_brief(recs_text=None):
    """
    Display an interactive voice player and speak the health brief.
    recs_text: pass your health_recs variable, or leave None to auto-use it.
    """
    # ── Get the text ─────────────────────────────────────────
    try:
        text = recs_text or health_recs or 'No health recommendations found. Please run Cell 7 first.'
    except NameError:
        text = 'No health recommendations found. Please run Cell 7 first.'

    # ── Build an intro ────────────────────────────────────────
    try:
        greeting = f'Good morning, {YOUR_NAME}! Here is your Neuro Band health brief for today.'
    except NameError:
        greeting = 'Good morning! Here is your Neuro Band health brief for today.'

    full_speech = greeting + ' ' + str(text)

    # ── Escape text for JS ────────────────────────────────────
    safe_text = json.dumps(full_speech)

    display(HTML(f"""
    <style>
      @import url('https://fonts.googleapis.com/css2?family=Syne:wght@700;800&family=DM+Sans:wght@300;400;500&display=swap');
      .vp * {{ font-family: 'DM Sans', sans-serif; box-sizing: border-box; }}
      .vp-btn {{
        border: none; cursor: pointer; border-radius: 50px;
        font-size: 1em; font-weight: 600; padding: 10px 22px;
        transition: all .2s; display: inline-flex;
        align-items: center; gap: 6px; outline: none;
      }}
      .vp-btn:hover {{ transform: translateY(-2px); filter: brightness(1.15); }}
      .vp-btn:active {{ transform: translateY(0); }}
      .vp-play  {{ background: linear-gradient(135deg,#667eea,#764ba2); color:#fff; }}
      .vp-pause {{ background: linear-gradient(135deg,#f093fb,#f5576c); color:#fff; }}
      .vp-stop  {{ background: rgba(255,255,255,.1); color:#ccc; border:1px solid rgba(255,255,255,.15); }}
      .vp-replay{{ background: linear-gradient(135deg,#11998e,#38ef7d); color:#fff; }}
      .vp-wave {{
        display: flex; align-items: center; gap: 3px;
        height: 24px; margin-left: 4px;
      }}
      .vp-bar {{
        width: 4px; border-radius: 2px;
        background: linear-gradient(to top, #667eea, #a8edea);
        animation: none;
      }}
      .vp-bar.active {{
        animation: wave .8s ease-in-out infinite;
      }}
      .vp-bar:nth-child(1){{ height:6px;  animation-delay:0s;    }}
      .vp-bar:nth-child(2){{ height:14px; animation-delay:.1s;   }}
      .vp-bar:nth-child(3){{ height:20px; animation-delay:.2s;   }}
      .vp-bar:nth-child(4){{ height:14px; animation-delay:.3s;   }}
      .vp-bar:nth-child(5){{ height:8px;  animation-delay:.15s;  }}
      .vp-bar:nth-child(6){{ height:18px; animation-delay:.25s;  }}
      .vp-bar:nth-child(7){{ height:12px; animation-delay:.05s;  }}
      @keyframes wave {{
        0%,100% {{ transform: scaleY(1);   }}
        50%      {{ transform: scaleY(2.5); }}
      }}
      .vp-speed {{
        background: rgba(255,255,255,.08);
        border: 1px solid rgba(255,255,255,.15);
        color: #ccc; border-radius: 8px;
        padding: 4px 10px; font-size: .8em; cursor: pointer;
      }}
    </style>

    <div class='vp' style='
      background: linear-gradient(145deg,#0f0c29,#302b63);
      border: 1px solid rgba(255,255,255,.08);
      border-radius: 18px;
      padding: 22px 24px;
      max-width: 580px;
      box-shadow: 0 20px 60px rgba(0,0,0,.5);
    '>

      <!-- Title -->
      <div style='display:flex;align-items:center;gap:10px;margin-bottom:16px;'>
        <span style='font-size:1.8em;'>🔊</span>
        <div>
          <div style='font-family:Syne,sans-serif;font-weight:800;font-size:1.1em;
                      color:#fff;letter-spacing:1px;'>HEALTH BRIEF PLAYER</div>
          <div style='color:#7b8cde;font-size:.75em;'>Neuro Band Voice System</div>
        </div>
        <!-- Animated wave bars -->
        <div class='vp-wave' style='margin-left:auto;'>
          <div class='vp-bar' id='b1'></div>
          <div class='vp-bar' id='b2'></div>
          <div class='vp-bar' id='b3'></div>
          <div class='vp-bar' id='b4'></div>
          <div class='vp-bar' id='b5'></div>
          <div class='vp-bar' id='b6'></div>
          <div class='vp-bar' id='b7'></div>
        </div>
      </div>

      <!-- Status bar -->
      <div id='vp-status' style='
        background: rgba(255,255,255,.05);
        border-radius: 8px; padding: 8px 12px;
        color: #a0a8ff; font-size: .8em; margin-bottom: 14px;
        border-left: 3px solid #667eea;
      '>⏸ Ready — click Play to hear your health brief</div>

      <!-- Text preview -->
      <div id='vp-text' style='
        background: rgba(255,255,255,.04);
        border-radius: 10px; padding: 10px 14px;
        color: #c5cae9; font-size: .82em; line-height: 1.7;
        max-height: 100px; overflow-y: auto;
        margin-bottom: 16px;
        border: 1px solid rgba(255,255,255,.06);
      '>{str(text).replace(chr(10), '<br>')}</div>

      <!-- Controls -->
      <div style='display:flex;gap:8px;flex-wrap:wrap;align-items:center;margin-bottom:14px;'>
        <button class='vp-btn vp-play'  id='vp-play'  onclick='vpPlay()'>▶ Play</button>
        <button class='vp-btn vp-pause' id='vp-pause' onclick='vpPause()' style='display:none;'>⏸ Pause</button>
        <button class='vp-btn vp-play'  id='vp-resume'onclick='vpResume()' style='display:none;'>▶ Resume</button>
        <button class='vp-btn vp-stop'  id='vp-stop'  onclick='vpStop()'>■ Stop</button>
        <button class='vp-btn vp-replay'onclick='vpReplay()'>↺ Replay</button>
      </div>

      <!-- Speed & Pitch sliders -->
      <div style='display:grid;grid-template-columns:1fr 1fr;gap:12px;'>
        <div>
          <label style='color:#7b8cde;font-size:.75em;letter-spacing:1px;'>SPEED
            <span id='rate-val' style='color:#fff;margin-left:6px;'>0.92×</span>
          </label>
          <input type='range' id='rate-slider' min='0.5' max='1.6' step='0.05' value='0.92'
                 oninput="window.nbVoiceSettings.rate=parseFloat(this.value);
                          document.getElementById('rate-val').textContent=parseFloat(this.value).toFixed(2)+'×';"
                 style='width:100%;accent-color:#667eea;cursor:pointer;'>
        </div>
        <div>
          <label style='color:#7b8cde;font-size:.75em;letter-spacing:1px;'>PITCH
            <span id='pitch-val' style='color:#fff;margin-left:6px;'>1.05</span>
          </label>
          <input type='range' id='pitch-slider' min='0.5' max='2.0' step='0.05' value='1.05'
                 oninput="window.nbVoiceSettings.pitch=parseFloat(this.value);
                          document.getElementById('pitch-val').textContent=parseFloat(this.value).toFixed(2);"
                 style='width:100%;accent-color:#f093fb;cursor:pointer;'>
        </div>
      </div>

    </div>

    <script>
    var VP_TEXT = {safe_text};

    function setWave(active) {{
      ['b1','b2','b3','b4','b5','b6','b7'].forEach(function(id) {{
        var el = document.getElementById(id);
        if (el) active ? el.classList.add('active') : el.classList.remove('active');
      }});
    }}

    function setStatus(msg, color) {{
      var s = document.getElementById('vp-status');
      if (s) {{ s.textContent = msg; s.style.color = color || '#a0a8ff'; }}
    }}

    function showBtn(play, pause, resume) {{
      document.getElementById('vp-play').style.display   = play   ? 'inline-flex' : 'none';
      document.getElementById('vp-pause').style.display  = pause  ? 'inline-flex' : 'none';
      document.getElementById('vp-resume').style.display = resume ? 'inline-flex' : 'none';
    }}

    function vpPlay() {{
      showBtn(false, true, false);
      setStatus('🔊 Speaking your health brief…', '#2ed573');
      setWave(true);
      window.nbSpeak(VP_TEXT, function() {{
        setStatus('✅ Done! Click Replay to hear again.', '#a0a8ff');
        setWave(false);
        showBtn(true, false, false);
      }});
    }}

    function vpPause() {{
      window.nbPause();
      showBtn(false, false, true);
      setStatus('⏸ Paused.', '#ffa502');
      setWave(false);
    }}

    function vpResume() {{
      window.nbResume();
      showBtn(false, true, false);
      setStatus('🔊 Resuming…', '#2ed573');
      setWave(true);
    }}

    function vpStop() {{
      window.nbStop();
      showBtn(true, false, false);
      setStatus('■ Stopped.', '#ff4757');
      setWave(false);
    }}

    function vpReplay() {{
      vpStop();
      setTimeout(vpPlay, 300);
    }}
    </script>
    """))
    print('\n✅ Voice player displayed above!')
    print('   Click ▶ Play to hear your health brief spoken aloud.')
    print('   Adjust Speed and Pitch sliders for your preferred voice style.')


# ── Launch the player ─────────────────────────────────────────
speak_health_brief()
print('\n👉 After listening, run Cell 8 for the full Morning Brief Dashboard!')

In [ ]:
# ============================================================
# CELL 8 — MORNING BRIEF DASHBOARD  (with built-in voice button)
# Replaces the old Cell 8 — has a 🔊 Speak Brief button built in
# ============================================================

from datetime import datetime, date, timedelta
from IPython.display import display, HTML

def generate_morning_brief_with_voice():
    data       = load_data()
    today_date = datetime.now().strftime('%A, %B %d, %Y')
    today_time = datetime.now().strftime('%I:%M %p')

    yest_cards  = data['flashcards'].get(yesterday_str, [])
    today_cards = data['flashcards'].get(today_str,     [])
    pending     = [t for t in data['todos'] if not t['done']]
    done_tasks  = [t for t in data['todos'] if  t['done']]
    yest_topics = list(set(c.get('topic','General') for c in yest_cards))

    # ── Weather block ─────────────────────────────────────────
    w = weather_info
    weather_html = f"""
    <div style='display:grid;grid-template-columns:repeat(4,1fr);gap:8px;margin:10px 0;'>
      <div style='background:rgba(255,255,255,.1);padding:10px;border-radius:8px;text-align:center;'>
        <div style='font-size:1.6em'>🌡️</div>
        <div style='font-size:1.1em;font-weight:700'>{w['temp_c']}°C</div>
        <div style='opacity:.6;font-size:.75em'>Temp (feels {w['feels_c']}°C)</div>
      </div>
      <div style='background:rgba(255,255,255,.1);padding:10px;border-radius:8px;text-align:center;'>
        <div style='font-size:1.6em'>💧</div>
        <div style='font-size:1.1em;font-weight:700'>{w['humidity']}%</div>
        <div style='opacity:.6;font-size:.75em'>Humidity</div>
      </div>
      <div style='background:rgba(255,255,255,.1);padding:10px;border-radius:8px;text-align:center;'>
        <div style='font-size:1.6em'>☀️</div>
        <div style='font-size:1.1em;font-weight:700'>{w['uv_index']}</div>
        <div style='opacity:.6;font-size:.75em'>UV Index</div>
      </div>
      <div style='background:rgba(255,255,255,.1);padding:10px;border-radius:8px;text-align:center;'>
        <div style='font-size:1.6em'>💨</div>
        <div style='font-size:1.1em;font-weight:700'>{w['wind_kmph']}</div>
        <div style='opacity:.6;font-size:.75em'>Wind km/h</div>
      </div>
    </div>
    <div style='text-align:center;opacity:.7;font-size:.85em;'>
      {w['desc']} &nbsp;|&nbsp; {w['min_c']}°C – {w['max_c']}°C today
    </div>"""

    # ── Flashcards block ──────────────────────────────────────
    if yest_cards:
        card_items = ''.join([
            f"<li style='margin:5px 0'>"
            f"<span style='background:rgba(102,126,234,.5);padding:1px 7px;border-radius:10px;font-size:.75em;'>"
            f"{c.get('topic','?')}</span> {c['question'][:55]}{'…' if len(c['question'])>55 else ''}</li>"
            for c in yest_cards[:6]
        ])
        extra    = f"<li style='opacity:.5'>…and {len(yest_cards)-6} more</li>" if len(yest_cards)>6 else ""
        fc_html  = f"<ul style='margin:6px 0;padding-left:18px;'>{card_items}{extra}</ul>"
    else:
        fc_html  = "<p style='opacity:.5;margin:6px 0;'>No flashcards recorded yesterday.</p>"

    # ── To-do block ──────────────────────────────────────────
    if pending:
        priority_order = {'🔴 High':0,'🟡 Medium':1,'🟢 Low':2}
        badge_colors   = {'🔴 High':'#ff4757','🟡 Medium':'#ffa502','🟢 Low':'#2ed573'}
        sorted_p = sorted(pending, key=lambda t: priority_order.get(t['priority'],1))
        todo_items = ''.join([
            f"<li style='margin:5px 0;'>"
            f"<span style='background:{badge_colors.get(t['priority'],'#888')};"
            f"padding:1px 7px;border-radius:10px;font-size:.75em;color:#fff;'>"
            f"{t['priority'].split(' ')[-1].upper()}</span> {t['task']}</li>"
            for t in sorted_p[:8]
        ])
        todo_html = f"<ul style='margin:6px 0;padding-left:18px;'>{todo_items}</ul>"
    else:
        todo_html = "<p style='opacity:.5;margin:6px 0;'>No pending tasks! All caught up.</p>"

    # ── Health & voice ────────────────────────────────────────
    health_text = str(health_recs).replace('\n','<br>') if health_recs else \
        'Run Cell 7 to generate health recommendations.'
    health_raw  = str(health_recs) if health_recs else ''

    try:
        greeting_js = f'Good morning, {YOUR_NAME}! Here is your Neuro Band health brief. '
    except Exception:
        greeting_js = 'Good morning! Here is your health brief. '

    import json as _json
    speech_js = _json.dumps(greeting_js + health_raw)

    # ── Full HTML ─────────────────────────────────────────────
    html = f"""
    <style>
      @import url('https://fonts.googleapis.com/css2?family=Syne:wght@400;600;800&family=DM+Sans:wght@300;400;500&display=swap');
      .nb8 * {{ font-family:'DM Sans',sans-serif; color:white; box-sizing:border-box; }}
      .nb8 h1,.nb8 h2 {{ font-family:'Syne',sans-serif; }}
      .nb8-sec {{ background:rgba(255,255,255,.06); border-radius:14px; padding:14px 16px; margin-bottom:14px; }}
      .nb8-lbl {{ font-size:.75em; letter-spacing:2px; opacity:.55; font-family:'Syne',sans-serif;
                  text-transform:uppercase; margin-bottom:6px; }}
      .speak-btn {{
        background: linear-gradient(135deg,#667eea,#764ba2);
        border:none; color:white; padding:10px 22px; border-radius:50px;
        font-size:.9em; font-weight:600; cursor:pointer;
        display:inline-flex; align-items:center; gap:8px;
        transition:all .2s; font-family:'DM Sans',sans-serif;
      }}
      .speak-btn:hover {{ transform:translateY(-2px); filter:brightness(1.15); }}
      .stop-btn {{
        background:rgba(255,255,255,.1); border:1px solid rgba(255,255,255,.2);
        color:#ccc; padding:10px 18px; border-radius:50px;
        font-size:.9em; cursor:pointer; transition:all .2s;
        font-family:'DM Sans',sans-serif;
      }}
      .stop-btn:hover {{ background:rgba(255,71,87,.3); }}
      .nb8-bar {{ width:4px; border-radius:2px;
                  background:linear-gradient(to top,#667eea,#a8edea);
                  animation:none; }}
      .nb8-bar.sp {{ animation:nb8wave .8s ease-in-out infinite; }}
      .nb8-bar:nth-child(1){{ height:5px;  animation-delay:0s;   }}
      .nb8-bar:nth-child(2){{ height:12px; animation-delay:.1s;  }}
      .nb8-bar:nth-child(3){{ height:18px; animation-delay:.2s;  }}
      .nb8-bar:nth-child(4){{ height:12px; animation-delay:.3s;  }}
      .nb8-bar:nth-child(5){{ height:7px;  animation-delay:.15s; }}
      @keyframes nb8wave {{ 0%,100%{{transform:scaleY(1)}} 50%{{transform:scaleY(2.4)}} }}
    </style>

    <div class='nb8' style='
        background:linear-gradient(145deg,#0f0c29 0%,#302b63 55%,#1a1a2e 100%);
        padding:28px 24px; border-radius:22px; max-width:720px;
        border:1px solid rgba(255,255,255,.08);
        box-shadow:0 30px 80px rgba(0,0,0,.6);
    '>

      <!-- HEADER -->
      <div style='text-align:center;margin-bottom:22px;
                  padding-bottom:18px;border-bottom:1px solid rgba(255,255,255,.1);'>
        <div style='font-size:2.8em;margin-bottom:4px;'>🧠⚡</div>
        <h1 style='margin:0;font-size:2em;font-weight:800;letter-spacing:3px;'>NEURO BAND</h1>
        <p style='margin:2px 0 0;opacity:.45;font-size:.8em;letter-spacing:4px;'>MORNING BRIEF SYSTEM</p>
        <div style='margin-top:12px;display:flex;justify-content:center;gap:24px;opacity:.7;font-size:.85em;'>
          <span>👋 Good morning, {YOUR_NAME}!</span>
          <span>📅 {today_date}</span>
          <span>⏰ {today_time}</span>
        </div>
        <div style='margin-top:10px;display:flex;justify-content:center;gap:16px;'>
          <span style='background:rgba(102,126,234,.3);padding:3px 12px;border-radius:20px;font-size:.8em;'>
            📚 {len(yest_cards)} cards yesterday</span>
          <span style='background:rgba(17,153,142,.3);padding:3px 12px;border-radius:20px;font-size:.8em;'>
            ✅ {len(pending)} tasks pending</span>
          <span style='background:rgba(255,71,87,.3);padding:3px 12px;border-radius:20px;font-size:.8em;'>
            🏆 {len(done_tasks)} tasks done</span>
        </div>
      </div>

      <!-- WEATHER -->
      <div class='nb8-sec'>
        <div class='nb8-lbl'>🌤️ Weather — {w['city']}</div>
        {weather_html}
      </div>

      <!-- FLASHCARDS -->
      <div class='nb8-sec'>
        <div class='nb8-lbl'>📚 Yesterday's Flashcards — {len(yest_cards)} cards
          | Topics: {', '.join(yest_topics) if yest_topics else 'None'}</div>
        {fc_html}
      </div>

      <!-- TO-DO -->
      <div class='nb8-sec'>
        <div class='nb8-lbl'>✅ To-Do — {len(pending)} pending / {len(done_tasks)} done</div>
        {todo_html}
      </div>

      <!-- HEALTH AI + VOICE -->
      <div style='
        background:rgba(102,126,234,.15);
        border:1px solid rgba(102,126,234,.35);
        border-radius:14px; padding:14px 16px; margin-bottom:14px;
      '>
        <div style='display:flex;align-items:center;justify-content:space-between;margin-bottom:10px;'>
          <div class='nb8-lbl' style='margin:0;'>🧬 AI Health Brief</div>
          <!-- Wave bars -->
          <div style='display:flex;align-items:center;gap:3px;height:20px;'>
            <div class='nb8-bar' id='nb8b1'></div>
            <div class='nb8-bar' id='nb8b2'></div>
            <div class='nb8-bar' id='nb8b3'></div>
            <div class='nb8-bar' id='nb8b4'></div>
            <div class='nb8-bar' id='nb8b5'></div>
          </div>
        </div>

        <div style='line-height:1.75;font-size:.88em;opacity:.9;margin-bottom:14px;'>
          {health_text}
        </div>

        <!-- Voice Controls -->
        <div style='display:flex;gap:8px;align-items:center;flex-wrap:wrap;
                    padding-top:12px;border-top:1px solid rgba(255,255,255,.08);'>
          <button class='speak-btn' id='nb8-play' onclick='nb8Play()'>
            🔊 Speak Brief
          </button>
          <button class='speak-btn' id='nb8-pause'
                  style='display:none;background:linear-gradient(135deg,#f093fb,#f5576c);'
                  onclick='nb8Pause()'>⏸ Pause</button>
          <button class='speak-btn' id='nb8-resume'
                  style='display:none;background:linear-gradient(135deg,#11998e,#38ef7d);'
                  onclick='nb8Resume()'>▶ Resume</button>
          <button class='stop-btn' onclick='nb8Stop()'>■ Stop</button>
          <span id='nb8-status'
                style='font-size:.78em;opacity:.5;font-family:DM Sans,sans-serif;'>
            Ready to speak
          </span>
        </div>
      </div>

      <!-- FOOTER -->
      <div style='text-align:center;opacity:.25;font-size:.7em;letter-spacing:2px;margin-top:4px;'>
        NEURO BAND MORNING BRIEF • {today_date}
      </div>
    </div>

    <script>
    var NB8_TEXT = {speech_js};

    function nb8Wave(on) {{
      ['nb8b1','nb8b2','nb8b3','nb8b4','nb8b5'].forEach(function(id) {{
        var el = document.getElementById(id);
        if (el) on ? el.classList.add('sp') : el.classList.remove('sp');
      }});
    }}
    function nb8Status(t,c){{
      var el=document.getElementById('nb8-status');
      if(el){{el.textContent=t;el.style.color=c||'rgba(255,255,255,.4)';}}
    }}
    function nb8Btns(play,pause,resume){{
      document.getElementById('nb8-play').style.display   = play  ?'inline-flex':'none';
      document.getElementById('nb8-pause').style.display  = pause ?'inline-flex':'none';
      document.getElementById('nb8-resume').style.display = resume?'inline-flex':'none';
    }}
    function nb8Play(){{
      nb8Btns(false,true,false); nb8Wave(true);
      nb8Status('Speaking…','#2ed573');
      if(window.nbSpeak){{
        window.nbSpeak(NB8_TEXT, function(){{
          nb8Btns(true,false,false); nb8Wave(false);
          nb8Status('Done! Click Speak Brief to replay.','rgba(255,255,255,.4)');
        }});
      }} else {{
        // Fallback if Cell 7-A was not run
        var u = new SpeechSynthesisUtterance(NB8_TEXT);
        u.rate=0.92; u.pitch=1.05;
        u.onend=function(){{nb8Btns(true,false,false);nb8Wave(false);nb8Status('Done!','rgba(255,255,255,.4)');}};
        window.speechSynthesis.cancel();
        window.speechSynthesis.speak(u);
      }}
    }}
    function nb8Pause(){{
      window.speechSynthesis.pause();
      nb8Btns(false,false,true); nb8Wave(false);
      nb8Status('Paused.','#ffa502');
    }}
    function nb8Resume(){{
      window.speechSynthesis.resume();
      nb8Btns(false,true,false); nb8Wave(true);
      nb8Status('Resuming…','#2ed573');
    }}
    function nb8Stop(){{
      window.speechSynthesis.cancel();
      nb8Btns(true,false,false); nb8Wave(false);
      nb8Status('Stopped.','rgba(255,255,255,.4)');
    }}
    </script>"""

    display(HTML(html))


generate_morning_brief_with_voice()

---
## 🔊 Voice Quick Reference

| Cell | Purpose | When to run |
|------|---------|-------------|
| **7-A** | Load voice engine into browser | Once per session, before 7-B |
| **7-B** | Standalone voice player | After Cell 7 (AI health) |
| **8**   | Full dashboard with built-in Speak button | Any time |

### 🎛️ Adjust Your Voice
In **Cell 7-A**, change `VOICE_SETTINGS`:
```python
VOICE_SETTINGS = {
    'rate'      : 0.85,              # Slower and clearer
    'pitch'     : 1.0,               # Natural pitch
    'volume'    : 1.0,               # Max volume
    'voice_name': 'Google UK English Female'  # British female voice
}
```

### 🌍 Best Free Voice Names by Browser
| Browser | Best voice name |
|---------|----------------|
| Chrome Windows | `Google US English` |
| Chrome Mac | `Samantha` |
| Chrome Linux | `Google UK English Female` |
| Edge | `Microsoft Aria Online` |

> ✅ **No API key, no cost, no limit** — uses your browser's built-in speech engine.